# Depop Product Crawler

This notebook demonstrates how to crawl Depop product pages and store them in Neo4j with price tracking.

## Features
- Extract product information (title, description, price, images)
- Track price history over time
- Store in graph database with relationships (Product -> Seller, Product -> Images, Product -> PriceHistory)
- Support for image galleries
- Context on garments (brand, size, condition, category, tags)


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent.parent / "src"))

from feed.platforms.depop import DepopAdapter
from feed.storage.neo4j_connection import get_connection
from feed.storage.product_storage import ProductStorage


## Initialize Connections


In [ ]:
# Connect to Neo4j
neo4j = get_connection()
print(f"Connected to: {neo4j.uri}")

# Initialize Depop adapter
depop = DepopAdapter(
    delay_min=2.0,
    delay_max=5.0,
    mock=False  # Set to True for testing without network requests
)

# Initialize product storage
product_storage = ProductStorage(neo4j)


## Crawl a Product

Example product URL: https://www.depop.com/products/krista_h-hot-pink-tights-thick-and/


In [ ]:
# Product URL to crawl
product_url = "https://www.depop.com/products/krista_h-hot-pink-tights-thick-and/"

# Fetch product data
product = depop.fetch_product(product_url)

if product:
    print(f"Product ID: {product.id}")
    print(f"Title: {product.title}")
    print(f"Price: {product.currency} {product.price}")
    print(f"Status: {product.status}")
    print(f"Brand: {product.brand}")
    print(f"Condition: {product.condition}")
    print(f"Size: {product.size}")
    print(f"Category: {product.category}")
    print(f"Tags: {product.tags}")
    print(f"Seller: {product.seller_username}")
    print(f"Likes: {product.likes_count}")
    print(f"Images: {len(product.image_urls)} images")
    for i, img_url in enumerate(product.image_urls[:3]):  # Show first 3
        print(f"  Image {i+1}: {img_url}")
    print(f"Description: {product.description[:200]}..." if len(product.description) > 200 else f"Description: {product.description}")
else:
    print("Failed to fetch product")


## Store Product in Neo4j


In [ ]:
if product:
    success = product_storage.store_product(product)
    if success:
        print("Product stored successfully!")
        print("\nGraph structure created:")
        print("  - Product node")
        print("  - Seller node (if seller exists)")
        print("  - Image nodes (one per image)")
        print("  - PriceHistory node (for price tracking)")
        print("  - Relationships: Product -> Seller, Product -> Images, Product -> PriceHistory")
    else:
        print("Failed to store product")


## Query Price History


In [ ]:
if product:
    price_history = product_storage.get_price_history(product.id, limit=10)
    print(f"Price history for {product.id}:")
    for entry in price_history:
        timestamp = entry.get('timestamp', 'N/A')
        price = entry.get('price', 'N/A')
        currency = entry.get('currency', 'USD')
        print(f"  {timestamp}: {currency} {price}")


## Check and Store Reddit Post

Check if a Reddit post exists in the database, and store it if missing.


In [ ]:
from feed.platforms.reddit import RedditAdapter
from feed.storage.thread_storage import store_thread
from feed.utils.reddit_url_extractor import extract_post_id_from_url, parse_reddit_url

# Post URL to check
post_url = "https://www.reddit.com/r/TaylorSwiftPictures/comments/1pt3bnb/loving_the_look/"

# Extract post ID
post_id = extract_post_id_from_url(post_url)
print(f"Post ID: {post_id}")

# Check if post exists
check_query = """
MATCH (p:Post {id: $post_id})
RETURN p.id as id, p.title as title, p.permalink as permalink
"""
result = neo4j.execute_read(check_query, parameters={"post_id": post_id})

if result:
    record = result[0]
    print(f"Post already exists in database:")
    print(f"  ID: {record['id']}")
    print(f"  Title: {record.get('title', 'N/A')}")
    print(f"  Permalink: {record.get('permalink', 'N/A')}")
else:
    print(f"Post not found in database. Fetching from Reddit...")
    
    # Fetch the post
    reddit = RedditAdapter()
    
    # Extract permalink from URL
    parsed = parse_reddit_url(post_url)
    if parsed and parsed.get('permalink'):
        permalink = parsed['permalink']
    else:
        # Fallback: try to extract from URL structure
        from urllib.parse import urlparse
        parsed_url = urlparse(post_url)
        path_parts = parsed_url.path.strip('/').split('/')
        if len(path_parts) >= 4 and path_parts[0] == 'r' and path_parts[2] == 'comments':
            permalink = '/' + '/'.join(path_parts[:4]) + '/'
        else:
            permalink = None
    
    if permalink:
        print(f"Fetching thread: {permalink}")
        post, comments, raw_post_data = reddit.fetch_thread(permalink)
        
        if post:
            print(f"Fetched post:")
            print(f"  ID: {post.id}")
            print(f"  Title: {post.title}")
            print(f"  Subreddit: {post.subreddit}")
            print(f"  Comments: {len(comments)}")
            
            # Extract images
            images = []
            
            # Extract images from post
            post_images = reddit.extract_all_images(post, raw_post_data)
            for img_url in post_images:
                images.append({
                    "url": img_url,
                    "source": "post",
                    "post_id": post.id,
                })
            
            # Extract images from comments
            comment_images = reddit.extract_images_from_comments(comments)
            images.extend(comment_images)
            
            print(f"  Images found: {len(images)}")
            
            # Store in Neo4j
            print(f"Storing in Neo4j...")
            try:
                store_thread(neo4j, post, comments, images, raw_post_data)
                print(f"Successfully stored post {post.id} in database!")
            except Exception as e:
                print(f"Error storing post: {e}")
                import traceback
                traceback.print_exc()
        else:
            print("Failed to fetch post from Reddit")
    else:
        print("Could not extract permalink from URL")


## Check and Store Another Reddit Post (with MFC links)

This post contains links to MyFigureCollection (MFC) data. We'll extract and store those links as well.


In [ ]:
from feed.platforms.reddit import RedditAdapter
from feed.storage.thread_storage import store_thread
from feed.utils.reddit_url_extractor import extract_post_id_from_url, parse_reddit_url
import re

# Post URL to check
post_url = "https://www.reddit.com/r/AnimeFigures/comments/1ptwzsa/kitagawa_marin_coreful_figure_veronica_ver_taito/"

# Extract post ID
post_id = extract_post_id_from_url(post_url)
print(f"Post ID: {post_id}")

# Check if post exists
check_query = """
MATCH (p:Post {id: $post_id})
RETURN p.id as id, p.title as title, p.permalink as permalink
"""
result = neo4j.execute_read(check_query, parameters={"post_id": post_id})

if result:
    record = result[0]
    print(f"Post already exists in database:")
    print(f"  ID: {record['id']}")
    print(f"  Title: {record.get('title', 'N/A')}")
    print(f"  Permalink: {record.get('permalink', 'N/A')}")
    
    # Check for MFC links in existing post
    mfc_query = """
    MATCH (p:Post {id: $post_id})
    OPTIONAL MATCH (p)-[:HAS_LINK]->(l:Link)
    WHERE l.url CONTAINS 'myfigurecollection.net'
    RETURN collect(l.url) as mfc_links
    """
    mfc_result = neo4j.execute_read(mfc_query, parameters={"post_id": post_id})
    if mfc_result and mfc_result[0].get('mfc_links'):
        print(f"  MFC links already stored: {len(mfc_result[0]['mfc_links'])}")
else:
    print(f"Post not found in database. Fetching from Reddit...")
    
    # Fetch the post
    reddit = RedditAdapter()
    
    # Extract permalink from URL
    parsed = parse_reddit_url(post_url)
    if parsed and parsed.get('permalink'):
        permalink = parsed['permalink']
    else:
        # Fallback: try to extract from URL structure
        from urllib.parse import urlparse
        parsed_url = urlparse(post_url)
        path_parts = parsed_url.path.strip('/').split('/')
        if len(path_parts) >= 4 and path_parts[0] == 'r' and path_parts[2] == 'comments':
            permalink = '/' + '/'.join(path_parts[:4]) + '/'
        else:
            permalink = None
    
    if permalink:
        print(f"Fetching thread: {permalink}")
        post, comments, raw_post_data = reddit.fetch_thread(permalink)
        
        if post:
            print(f"Fetched post:")
            print(f"  ID: {post.id}")
            print(f"  Title: {post.title}")
            print(f"  Subreddit: {post.subreddit}")
            print(f"  Comments: {len(comments)}")
            
            # Extract images
            images = []
            
            # Extract images from post
            post_images = reddit.extract_all_images(post, raw_post_data)
            for img_url in post_images:
                images.append({
                    "url": img_url,
                    "source": "post",
                    "post_id": post.id,
                })
            
            # Extract images from comments
            comment_images = reddit.extract_images_from_comments(comments)
            images.extend(comment_images)
            
            print(f"  Images found: {len(images)}")
            
            # Extract MFC links from post and comments
            mfc_links = []
            mfc_pattern = r'https?://(?:www\.)?myfigurecollection\.net/[^\s\)]+'
            
            # Check post selftext
            if post.selftext:
                found_links = re.findall(mfc_pattern, post.selftext)
                mfc_links.extend(found_links)
            
            # Check comments
            for comment in comments:
                if comment.body:
                    found_links = re.findall(mfc_pattern, comment.body)
                    for link in found_links:
                        mfc_links.append({
                            "url": link,
                            "source": "comment",
                            "comment_id": comment.id,
                        })
            
            # Deduplicate MFC links
            unique_mfc_links = list(dict.fromkeys([link if isinstance(link, str) else link['url'] for link in mfc_links]))
            print(f"  MFC links found: {len(unique_mfc_links)}")
            for link in unique_mfc_links[:5]:  # Show first 5
                print(f"    - {link}")
            
            # Store in Neo4j
            print(f"Storing in Neo4j...")
            try:
                store_thread(neo4j, post, comments, images, raw_post_data)
                print(f"Successfully stored post {post.id} in database!")
                
                # Store MFC links as Link nodes
                if unique_mfc_links:
                    print(f"Storing {len(unique_mfc_links)} MFC links...")
                    for link_url in unique_mfc_links:
                        link_query = """
                        MATCH (p:Post {id: $post_id})
                        MERGE (l:Link {url: $link_url})
                        ON CREATE SET l.created_at = datetime()
                        MERGE (p)-[:HAS_LINK]->(l)
                        """
                        neo4j.execute_write(
                            link_query,
                            parameters={"post_id": post.id, "link_url": link_url}
                        )
                    print(f"Stored MFC links successfully!")
            except Exception as e:
                print(f"Error storing post: {e}")
                import traceback
                traceback.print_exc()
        else:
            print("Failed to fetch post from Reddit")
    else:
        print("Could not extract permalink from URL")


## Check and Store Post with Image Source Detection

This post has two distinct images in the main post. We'll extract them and try to identify their sources.


In [ ]:
from feed.platforms.reddit import RedditAdapter
from feed.storage.thread_storage import store_thread
from feed.utils.reddit_url_extractor import extract_post_id_from_url, parse_reddit_url
import re
from urllib.parse import urlparse

# Post URL to check
post_url = "https://www.reddit.com/r/ClassroomOfTheElite/comments/1puhf44/the_true_definition_of_downgrade/"

# Extract post ID
post_id = extract_post_id_from_url(post_url)
print(f"Post ID: {post_id}")

# Check if post exists
check_query = """
MATCH (p:Post {id: $post_id})
RETURN p.id as id, p.title as title, p.permalink as permalink, p.url as url
"""
result = neo4j.execute_read(check_query, parameters={"post_id": post_id})

if result:
    record = result[0]
    print(f"Post already exists in database:")
    print(f"  ID: {record['id']}")
    print(f"  Title: {record.get('title', 'N/A')}")
    print(f"  URL: {record.get('url', 'N/A')}")
    
    # Get images for this post
    images_query = """
    MATCH (p:Post {id: $post_id})-[:HAS_IMAGE]->(img:Image)
    RETURN img.url as url, img.source as source
    ORDER BY img.created_at ASC
    """
    images_result = neo4j.execute_read(images_query, parameters={"post_id": post_id})
    if images_result:
        print(f"  Images stored: {len(images_result)}")
        for img in images_result:
            print(f"    - {img.get('url', 'N/A')} (source: {img.get('source', 'N/A')})")
else:
    print(f"Post not found in database. Fetching from Reddit...")
    
    # Fetch the post
    reddit = RedditAdapter()
    
    # Extract permalink from URL
    parsed = parse_reddit_url(post_url)
    if parsed and parsed.get('permalink'):
        permalink = parsed['permalink']
    else:
        from urllib.parse import urlparse
        parsed_url = urlparse(post_url)
        path_parts = parsed_url.path.strip('/').split('/')
        if len(path_parts) >= 4 and path_parts[0] == 'r' and path_parts[2] == 'comments':
            permalink = '/' + '/'.join(path_parts[:4]) + '/'
        else:
            permalink = None
    
    if permalink:
        print(f"Fetching thread: {permalink}")
        post, comments, raw_post_data = reddit.fetch_thread(permalink)
        
        if post:
            print(f"Fetched post:")
            print(f"  ID: {post.id}")
            print(f"  Title: {post.title}")
            print(f"  Subreddit: {post.subreddit}")
            print(f"  Comments: {len(comments)}")
            print(f"  Post URL: {post.url}")
            
            # Extract images from post
            images = []
            post_images = reddit.extract_all_images(post, raw_post_data)
            
            print(f"\n=== IMAGE ANALYSIS ===")
            print(f"Main post images found: {len(post_images)}")
            
            # Check if it's a gallery post (multiple images)
            if raw_post_data and raw_post_data.get('is_gallery'):
                print("  Type: Gallery post (multiple images)")
                gallery_items = raw_post_data.get('gallery_data', {}).get('items', [])
                print(f"  Gallery items: {len(gallery_items)}")
            
            # Analyze each image
            for i, img_url in enumerate(post_images, 1):
                print(f"\n  Image {i}:")
                print(f"    URL: {img_url}")
                
                # Try to identify image source domain
                parsed_img = urlparse(img_url)
                domain = parsed_img.netloc
                print(f"    Domain: {domain}")
                
                # Check for common image hosting patterns
                if 'i.redd.it' in domain:
                    print(f"    Source: Reddit image hosting")
                elif 'imgur.com' in domain:
                    print(f"    Source: Imgur")
                elif 'redd.it' in domain:
                    print(f"    Source: Reddit short link")
                else:
                    print(f"    Source: External ({domain})")
                
                images.append({
                    "url": img_url,
                    "source": "post",
                    "post_id": post.id,
                    "image_index": i,
                })
            
            # Extract images from comments
            comment_images = reddit.extract_images_from_comments(comments)
            print(f"\nComment images found: {len(comment_images)}")
            for img_data in comment_images:
                images.append(img_data)
            
            print(f"\nTotal images: {len(images)}")
            
            # Try to find image sources in comments (people often post source links)
            print(f"\n=== SEARCHING FOR IMAGE SOURCES IN COMMENTS ===")
            source_patterns = [
                r'(?:source|sauce|origin|from|credit)[:;]?\s*(https?://[^\s\)]+)',
                r'(?:image|pic|photo)[:;]?\s*(?:from|source|via)[:;]?\s*(https?://[^\s\)]+)',
                r'https?://(?:www\.)?(?:twitter|pixiv|deviantart|artstation|instagram|pinterest)\.com/[^\s\)]+',
            ]
            
            found_sources = []
            for comment in comments:
                if comment.body:
                    for pattern in source_patterns:
                        matches = re.findall(pattern, comment.body, re.IGNORECASE)
                        for match in matches:
                            if match not in found_sources:
                                found_sources.append(match)
                                print(f"  Found source link in comment {comment.id}: {match}")
            
            # Store in Neo4j
            print(f"\n=== STORING IN NEO4J ===")
            try:
                store_thread(neo4j, post, comments, images, raw_post_data)
                print(f"Successfully stored post {post.id} in database!")
                
                # Store source links if found
                if found_sources:
                    print(f"Storing {len(found_sources)} source links...")
                    for source_url in found_sources:
                        link_query = """
                        MATCH (p:Post {id: $post_id})
                        MERGE (l:Link {url: $link_url})
                        ON CREATE SET l.created_at = datetime(), l.link_type = 'image_source'
                        MERGE (p)-[:HAS_LINK {link_type: 'image_source'}]->(l)
                        """
                        neo4j.execute_write(
                            link_query,
                            parameters={"post_id": post.id, "link_url": source_url}
                        )
                    print(f"Stored source links successfully!")
                
                # Note about similarity search
                if len(post_images) >= 2:
                    print(f"\n=== SIMILARITY SEARCH NOTE ===")
                    print(f"This post has {len(post_images)} distinct images in the main post.")
                    print(f"These can be used for similarity search to find related content:")
                    for i, img_url in enumerate(post_images, 1):
                        print(f"  Image {i}: {img_url}")
                    print(f"\nYou can use these images for:")
                    print(f"  - Reverse image search")
                    print(f"  - Visual similarity matching")
                    print(f"  - Finding related posts with similar images")
                    
            except Exception as e:
                print(f"Error storing post: {e}")
                import traceback
                traceback.print_exc()
        else:
            print("Failed to fetch post from Reddit")
    else:
        print("Could not extract permalink from URL")


## Query Images for Similarity Search

After storing posts with multiple images, you can query them for similarity search.


## Fapello Profile Crawler

Crawl Fapello profiles and store creators, handles, and media in Neo4j knowledge graph.


## Garment Style Detection and Product Matching

This section demonstrates the garment style detection benchmark task:
1. Analyze an image to identify garment style
2. Create/update garment style ontology
3. Match style to e-commerce products
4. Set up price tracking


In [ ]:
from feed.services.garment_matcher import GarmentMatcher
from feed.storage.neo4j_connection import get_connection

# Connect to Neo4j
neo4j = get_connection()

# Initialize garment matcher
matcher = GarmentMatcher(neo4j)

# Example: Analyze the sheer panel yoga pants image
# This would come from CV model in production
image_analysis = {
    "garment_type": "Yoga Pants",
    "color": "Black",
    "panel_type": "Sheer Panel",
    "fit": "Compression",
    "length": "Full Length",
    "material": "Polyester/Spandex",
    "features": [
        "sheer_panel_thigh",
        "sheer_panel_calf",
        "vertical_panels",
        "compression_fit",
        "full_length",
        "black_color"
    ]
}

# Image URL (from Reddit post or other source)
image_url = "https://i.redd.it/example-sheer-panel-yoga-pants.jpg"
post_id = "1puhf44"  # Example post ID

print("Analyzing image and matching to products...")
result = matcher.analyze_and_match_image(
    image_url=image_url,
    image_analysis=image_analysis,
    post_id=post_id
)

print(f"\n=== RESULTS ===")
print(f"Style ID: {result['style_id']}")
print(f"Style Name: {result['style_name']}")
print(f"\nMatched Products: {len(result['matched_products'])}")
for product in result['matched_products'][:5]:
    print(f"  - {product['title']}: {product['currency']} {product['price']} (confidence: {product['confidence']:.2f})")
    print(f"    URL: {product['url']}")

print(f"\n=== MARKETPLACE SEARCH LINKS ===")
for platform, link in result['search_links'].items():
    print(f"  {platform}: {link}")

print(f"\n=== PRICE TRACKING SETUP ===")
print(f"Products set up for tracking: {result['tracking_setup']['tracked_count']}")


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent.parent / "src"))

from feed.platforms.fapello import FapelloAdapter
from feed.storage.neo4j_connection import get_connection
from feed.models.creator import Creator
from feed.models.handle import Handle
from feed.models.media import ImageMedia
from datetime import datetime
import re


In [ ]:
def store_fapello_profile(neo4j, profile_url: str, limit: int = 100):
    """
    Crawl a Fapello profile and store Creator, Handle, Platform, and Media nodes in Neo4j.
    
    Args:
        neo4j: Neo4j connection
        profile_url: Fapello profile URL (e.g., https://fapello.com/sjokz/)
        limit: Maximum number of media items to fetch
    """
    fapello = FapelloAdapter(delay_min=2.0, delay_max=5.0, mock=False)
    
    # Extract username from URL
    match = re.search(r'fapello\.com/([^/]+)/?', profile_url)
    if not match:
        print(f"Could not extract username from URL: {profile_url}")
        return
    username = match.group(1)
    
    print(f"Fetching profile: {profile_url}")
    print(f"Username: {username}")
    
    # Fetch profile metadata
    metadata = fapello.fetch_source_metadata(profile_url)
    if metadata:
        print(f"Profile: {metadata.name}")
        print(f"Media count: {metadata.subscribers if hasattr(metadata, 'subscribers') else 'N/A'}")
        print(f"Likes: {metadata.subscribers if hasattr(metadata, 'subscribers') else 'N/A'}")
    
    # Fetch all media items
    print(f"\nFetching media items (limit: {limit})...")
    media_items = fapello.fetch_media(profile_url, limit=limit)
    print(f"Found {len(media_items)} media items")
    
    # Create Platform node for Fapello
    platform_query = """
    MERGE (p:Platform {slug: $slug})
    ON CREATE SET 
        p.name = $name,
        p.slug = $slug,
        p.icon_url = $icon_url,
        p.created_at = datetime()
    RETURN p
    """
    neo4j.execute_write(
        platform_query,
        parameters={
            "slug": "fapello",
            "name": "Fapello",
            "icon_url": "https://fapello.com/assets/favicon/favicon-32x32.png"
        }
    )
    print("Created/updated Platform node: Fapello")
    
    # Extract creator name from profile (try to get full name)
    # For now, use username as slug and try to extract name from metadata
    creator_name = username.title()  # Default to capitalized username
    if metadata and hasattr(metadata, 'description'):
        # Try to extract name from description
        desc = metadata.description
        if desc and 'Fapello profile:' in desc:
            # Extract name from description like "Fapello profile: Sjokz (1712 media)"
            name_match = re.search(r'profile:\s*([^(]+)', desc)
            if name_match:
                creator_name = name_match.group(1).strip()
    
    # Create Creator node
    creator_query = """
    MERGE (c:Creator {slug: $slug})
    ON CREATE SET 
        c.uuid = randomUUID(),
        c.name = $name,
        c.slug = $slug,
        c.created_at = datetime(),
        c.updated_at = datetime()
    ON MATCH SET
        c.updated_at = datetime()
    RETURN c.uuid as uuid, c.slug as slug
    """
    creator_result = neo4j.execute_write(
        creator_query,
        parameters={
            "slug": username,
            "name": creator_name
        }
    )
    if creator_result:
        creator_uuid = creator_result[0]['uuid']
        print(f"Created/updated Creator node: {creator_name} (slug: {username}, uuid: {creator_uuid})")
    
    # Create Handle node for Fapello profile
    handle_query = """
    MATCH (c:Creator {slug: $creator_slug})
    MERGE (h:Handle {profile_url: $profile_url})
    ON CREATE SET
        h.uuid = randomUUID(),
        h.username = $username,
        h.display_name = $display_name,
        h.profile_url = $profile_url,
        h.created_at = datetime(),
        h.updated_at = datetime()
    ON MATCH SET
        h.updated_at = datetime()
    WITH h, c
    MERGE (c)-[r:OWNS_HANDLE]->(h)
    ON CREATE SET
        r.status = 'Active',
        r.verified = true,
        r.confidence = 'High',
        r.discovered_at = datetime(),
        r.created_at = datetime()
    WITH h
    MATCH (p:Platform {slug: 'fapello'})
    MERGE (h)-[:ON_PLATFORM]->(p)
    RETURN h.uuid as uuid
    """
    handle_result = neo4j.execute_write(
        handle_query,
        parameters={
            "creator_slug": username,
            "username": username,
            "display_name": creator_name,
            "profile_url": profile_url.rstrip('/')
        }
    )
    if handle_result:
        handle_uuid = handle_result[0]['uuid']
        print(f"Created/updated Handle node: {username} (uuid: {handle_uuid})")
    
    # Store all media items
    print(f"\nStoring {len(media_items)} media items...")
    stored_count = 0
    
    for i, media in enumerate(media_items, 1):
        if i % 50 == 0:
            print(f"  Progress: {i}/{len(media_items)}")
        
        # Create Media node
        media_dict = media.to_neo4j_dict()
        
        media_query = """
        MERGE (m:Media {source_url: $source_url})
        ON CREATE SET
            m.uuid = randomUUID(),
            m.title = $title,
            m.source_url = $source_url,
            m.publish_date = datetime({epochSeconds: $publish_epoch}),
            m.thumbnail_url = $thumbnail_url,
            m.media_type = $media_type,
            m.created_at = datetime(),
            m.updated_at = datetime()
        ON MATCH SET
            m.updated_at = datetime()
        WITH m
        WHERE m.media_type = 'Image'
        SET m:Image
        WITH m
        MATCH (h:Handle {profile_url: $profile_url})
        MERGE (h)-[r:PUBLISHED]->(m)
        ON CREATE SET
            r.published_at = datetime({epochSeconds: $publish_epoch}),
            r.created_at = datetime()
        WITH m
        MATCH (p:Platform {slug: 'fapello'})
        MERGE (m)-[:SOURCED_FROM]->(p)
        RETURN m.uuid as uuid
        """
        
        publish_epoch = int(media.publish_date.timestamp()) if media.publish_date else int(datetime.utcnow().timestamp())
        
        try:
            result = neo4j.execute_write(
                media_query,
                parameters={
                    "source_url": media.source_url,
                    "title": media.title or f"{username} - {i}",
                    "publish_epoch": publish_epoch,
                    "thumbnail_url": media.thumbnail_url,
                    "media_type": media.media_type.value,
                    "profile_url": profile_url.rstrip('/')
                }
            )
            if result:
                stored_count += 1
        except Exception as e:
            print(f"  Error storing media {i}: {e}")
            continue
    
    print(f"\nSuccessfully stored {stored_count}/{len(media_items)} media items")
    print(f"\nKnowledge graph structure created:")
    print(f"  - Creator: {creator_name} (slug: {username})")
    print(f"  - Handle: {username} on Fapello")
    print(f"  - Platform: Fapello")
    print(f"  - Media: {stored_count} Image nodes")
    print(f"  - Relationships: Creator -> Handle -> Platform, Handle -> Media, Media -> Platform")


In [ ]:
# Crawl Sjokz profile
profile_url = "https://fapello.com/sjokz/"

# Connect to Neo4j
neo4j = get_connection()
print(f"Connected to: {neo4j.uri}\n")

# Store the profile and all media
store_fapello_profile(neo4j, profile_url, limit=100)  # Start with 100, can increase later


In [ ]:
# Query the stored data
query = """
MATCH (c:Creator {slug: 'sjokz'})-[:OWNS_HANDLE]->(h:Handle)-[:ON_PLATFORM]->(p:Platform)
OPTIONAL MATCH (h)-[:PUBLISHED]->(m:Media)
RETURN 
    c.name as creator_name,
    c.slug as creator_slug,
    h.username as handle_username,
    p.name as platform_name,
    count(DISTINCT m) as media_count
"""
result = neo4j.execute_read(query)
if result:
    for record in result:
        print(f"Creator: {record['creator_name']} (slug: {record['creator_slug']})")
        print(f"Handle: {record['handle_username']} on {record['platform_name']}")
        print(f"Media count: {record['media_count']}")


## Extract Additional Profile Metadata

The Fapello profile page contains additional metadata like:
- Full name: "Eefje Depoortere"
- Aliases: "eefjah, sjokz"
- Social links: Instagram, Twitter, CamSoda
- Media count: 1712
- Likes: 472

We can enhance the crawler to extract and store this information.


In [ ]:
from bs4 import BeautifulSoup
import requests

def extract_fapello_profile_metadata(profile_url: str):
    """Extract detailed metadata from Fapello profile page."""
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"
    }
    
    response = requests.get(profile_url, headers=headers, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')
    
    metadata = {
        "full_name": None,
        "aliases": [],
        "social_links": {},
        "media_count": 0,
        "likes_count": 0,
        "avatar_url": None,
    }
    
    # Extract full name from h2 tag
    h2 = soup.find('h2')
    if h2:
        metadata["full_name"] = h2.get_text(strip=True)
    
    # Extract aliases from paragraph after h2
    # Pattern: "Eefje Depoortere, eefjah, sjokz"
    p_tags = soup.find_all('p')
    for p in p_tags:
        text = p.get_text(strip=True)
        if ',' in text and len(text) < 200:  # Likely the aliases line
            parts = [part.strip() for part in text.split(',')]
            if len(parts) > 1:
                metadata["aliases"] = parts
                break
    
    # Extract social links
    social_links = soup.find_all('a', href=True)
    for link in social_links:
        href = link.get('href', '')
        text = link.get_text(strip=True)
        
        if 'instagram.com' in href:
            metadata["social_links"]["instagram"] = href
        elif 'twitter.com' in href or 'x.com' in href:
            metadata["social_links"]["twitter"] = href
        elif 'camsoda.com' in href:
            metadata["social_links"]["camsoda"] = href
    
    # Extract media count and likes
    text = soup.get_text()
    media_match = re.search(r'(\d+)\s+Media', text, re.IGNORECASE)
    if media_match:
        metadata["media_count"] = int(media_match.group(1))
    
    likes_match = re.search(r'(\d+)\s+Likes', text, re.IGNORECASE)
    if likes_match:
        metadata["likes_count"] = int(likes_match.group(1))
    
    # Extract avatar URL
    img = soup.find('img', class_=lambda x: x and 'rounded-full' in x if x else False)
    if img:
        src = img.get('src') or img.get('data-src')
        if src:
            if src.startswith('//'):
                src = 'https:' + src
            elif src.startswith('/'):
                src = 'https://fapello.com' + src
            elif not src.startswith('http'):
                src = f"https://fapello.com/content/{profile_url.split('/')[-2]}/{src}"
            metadata["avatar_url"] = src
    
    return metadata

# Extract metadata for Sjokz
sjokz_metadata = extract_fapello_profile_metadata("https://fapello.com/sjokz/")
print("Extracted metadata:")
for key, value in sjokz_metadata.items():
    print(f"  {key}: {value}")


In [ ]:
# Update Creator node with full metadata
if sjokz_metadata["full_name"]:
    update_query = """
    MATCH (c:Creator {slug: 'sjokz'})
    SET c.name = $full_name,
        c.bio = $bio,
        c.avatar_url = $avatar_url,
        c.updated_at = datetime()
    RETURN c
    """
    
    bio_parts = []
    if sjokz_metadata["full_name"]:
        bio_parts.append(f"Full name: {sjokz_metadata['full_name']}")
    if sjokz_metadata["aliases"]:
        bio_parts.append(f"Aliases: {', '.join(sjokz_metadata['aliases'])}")
    if sjokz_metadata["social_links"]:
        links = [f"{platform}: {url}" for platform, url in sjokz_metadata["social_links"].items()]
        bio_parts.append(f"Social links: {', '.join(links)}")
    
    bio = " | ".join(bio_parts) if bio_parts else None
    
    neo4j.execute_write(
        update_query,
        parameters={
            "full_name": sjokz_metadata["full_name"],
            "bio": bio,
            "avatar_url": sjokz_metadata.get("avatar_url")
        }
    )
    print(f"Updated Creator node with full name: {sjokz_metadata['full_name']}")
    print(f"Bio: {bio}")


In [ ]:
# Now crawl ALL media (1712 items) - this will take a while!
# Uncomment to run the full crawl

# print("Starting full media crawl (1712 items)...")
# store_fapello_profile(neo4j, "https://fapello.com/sjokz/", limit=2000)  # Get all 1712+ items


In [ ]:
# Query posts with multiple images (good for similarity search)
multi_image_query = """
MATCH (p:Post)-[r:HAS_IMAGE]->(img:Image)
WHERE r.image_index IS NOT NULL
WITH p, collect(img.url) as images, count(img) as image_count
WHERE image_count >= 2
RETURN p.id as post_id, p.title as title, p.subreddit as subreddit, images, image_count
ORDER BY image_count DESC
LIMIT 10
"""

result = neo4j.execute_read(multi_image_query)
print("Posts with multiple images (suitable for similarity search):")
print("=" * 80)
for record in result:
    print(f"\nPost ID: {record['post_id']}")
    print(f"  Title: {record.get('title', 'N/A')}")
    print(f"  Subreddit: r/{record.get('subreddit', 'N/A')}")
    print(f"  Image count: {record['image_count']}")
    print(f"  Images:")
    for i, img_url in enumerate(record['images'], 1):
        print(f"    {i}. {img_url}")


In [ ]:
# Reddit/Taylor Swift tracking has been moved to taylor_swift_reddit_tracker.ipynb
# This notebook is for Depop product crawling only


## Reverse Image Search Tooling

This section demonstrates how to use the reverse image search service for:
1. **Checking if an image has been crawled** - Fast lookup to avoid duplicate work
2. **Finding original source** - Track where images first appeared
3. **Finding exact matches** - Discover all occurrences of an image across platforms
4. **Visual similarity search** - Find visually similar images using CLIP embeddings

### Use Cases:
- **Feed Management**: Before crawling a new post, check if its images are already in the database
- **Deduplication**: Identify duplicate images across different posts/products
- **Source Attribution**: Find the original source of an image
- **Content Discovery**: Find related images using visual similarity


In [ ]:
from feed.services.reverse_image_search import ReverseImageSearch
from feed.storage.neo4j_connection import get_connection

# Connect to Neo4j
neo4j = get_connection()

# Initialize reverse image search service
# This will automatically set up vector search if CLIP is available
reverse_search = ReverseImageSearch(
    neo4j=neo4j,
    enable_vector_search=True,  # Enable CLIP-based visual similarity
    enable_external_apis=False   # Set to True to enable external APIs (TinEye, etc.)
)

print("Reverse image search service initialized!")
print(f"Vector search enabled: {reverse_search.enable_vector_search}")


### Example 1: Check if Image Has Been Crawled

Before processing a new image, check if we've already seen it. This is useful for:
- Avoiding duplicate work in feed crawlers
- Detecting reposts
- Tracking image reuse across platforms


In [ ]:
# Example: Check if an image URL has been crawled
image_url = "https://i.redd.it/example-image.jpg"  # Replace with actual image URL

result = reverse_search.check_if_crawled(image_url)

print(f"Image URL: {image_url}")
print(f"Found in database: {result.found}")
print(f"Number of matches: {len(result.matches)}")

if result.found:
    print("\nMatches found:")
    for match in result.matches:
        print(f"  - Match type: {match.match_type}")
        print(f"    Confidence: {match.confidence:.2f}")
        if match.source_post_id:
            print(f"    Source: Reddit post {match.source_post_id}")
        if match.source_comment_id:
            print(f"    Source: Reddit comment {match.source_comment_id}")
        if match.source_product_id:
            print(f"    Source: Product {match.source_product_id}")
else:
    print("\nImage not found in database. Safe to crawl.")


### Example 2: Find Original Source

Find where an image first appeared in our database. Useful for:
- Attribution tracking
- Finding the earliest occurrence
- Understanding image propagation


In [ ]:
# Example: Find original source of an image
image_url = "https://i.redd.it/example-image.jpg"  # Replace with actual image URL

result = reverse_search.find_original_source(image_url, include_external=False)

print(f"Image URL: {image_url}")
print(f"Source found: {result.found}")

if result.found and result.matches:
    earliest_match = result.matches[0]  # Results are sorted by time
    print(f"\nEarliest occurrence:")
    print(f"  URL: {earliest_match.image_url}")
    if earliest_match.source_post_id:
        print(f"  Source: Reddit post {earliest_match.source_post_id}")
    if earliest_match.metadata and earliest_match.metadata.get("earliest_time"):
        print(f"  First seen: {earliest_match.metadata['earliest_time']}")


### Example 3: Find Exact Matches

Find all exact and near-exact matches of an image. Uses:
- SHA-256 hash for exact duplicates
- dHash for near-duplicates (cropped, resized, etc.)
- CLIP embeddings for visually similar images


In [ ]:
# Example: Find all exact matches
image_url = "https://i.redd.it/example-image.jpg"  # Replace with actual image URL

result = reverse_search.find_exact_matches(image_url, include_external=False)

print(f"Image URL: {image_url}")
print(f"Matches found: {len(result.matches)}")

if result.hashes:
    print(f"\nImage hashes:")
    print(f"  SHA-256: {result.hashes.get('sha256', 'N/A')[:16]}...")
    print(f"  dHash: {result.hashes.get('dhash', 'N/A')}")

if result.matches:
    print(f"\nMatches by type:")
    matches_by_type = {}
    for match in result.matches:
        match_type = match.match_type
        if match_type not in matches_by_type:
            matches_by_type[match_type] = []
        matches_by_type[match_type].append(match)
    
    for match_type, matches in matches_by_type.items():
        print(f"\n  {match_type.upper()} matches ({len(matches)}):")
        for match in matches[:5]:  # Show first 5
            print(f"    - {match.image_url}")
            print(f"      Confidence: {match.confidence:.2f}")
            if match.source_post_id:
                print(f"      From post: {match.source_post_id}")


### Example 4: Index Images for Future Search

After crawling an image, index it for future reverse search. This stores:
- Hash values in Neo4j (for fast duplicate detection)
- CLIP embeddings in Valkey (for visual similarity search)


In [ ]:
# Example: Index an image after crawling
# This is typically done automatically during the crawl process,
# but you can also index images manually

image_url = "https://i.redd.it/example-image.jpg"  # Replace with actual image URL

# Check if image exists in Neo4j first
check_query = """
MATCH (img:Image {url: $url})
RETURN img.url as url
LIMIT 1
"""
exists = neo4j.execute_read(check_query, parameters={"url": image_url})

if exists:
    print(f"Indexing image: {image_url}")
    success = reverse_search.index_image(image_url)
    if success:
        print("Image indexed successfully!")
        print("  - Hash values stored in Neo4j")
        if reverse_search.enable_vector_search:
            print("  - CLIP embedding stored in Valkey")
    else:
        print("Failed to index image")
else:
    print(f"Image {image_url} not found in Neo4j. Store it first using store_thread() or product_storage.store_product()")


### Example 5: Integration with Feed Crawler

Here's how to integrate reverse image search into your feed crawler workflow:


In [ ]:
# Example: Check images before crawling a Reddit post
from feed.platforms.reddit import RedditAdapter
from feed.storage.thread_storage import store_thread

# Post URL to check
post_url = "https://www.reddit.com/r/example/comments/abc123/example_post/"

# Initialize Reddit adapter
reddit = RedditAdapter()

# Fetch post data
post, comments, raw_post_data = reddit.fetch_thread(post_url)

if post:
    # Extract images from post
    post_images = reddit.extract_all_images(post, raw_post_data)
    
    print(f"Post: {post.title}")
    print(f"Images found: {len(post_images)}")
    
    # Check each image before processing
    new_images = []
    duplicate_images = []
    
    for img_url in post_images:
        result = reverse_search.check_if_crawled(img_url)
        
        if result.found:
            duplicate_images.append({
                "url": img_url,
                "matches": result.matches
            })
            print(f"  [DUPLICATE] {img_url}")
            print(f"    Already seen in {len(result.matches)} location(s)")
        else:
            new_images.append(img_url)
            print(f"  [NEW] {img_url}")
    
    print(f"\nSummary:")
    print(f"  New images: {len(new_images)}")
    print(f"  Duplicate images: {len(duplicate_images)}")
    
    # Only store if there are new images, or if you want to track all occurrences
    if new_images or True:  # Set to False to skip posts with only duplicates
        print(f"\nStoring post in Neo4j...")
        images = [{"url": url, "source": "post", "post_id": post.id} for url in post_images]
        store_thread(neo4j, post, comments, images, raw_post_data)
        
        # Index new images for future search
        for img_url in new_images:
            reverse_search.index_image(img_url)
        
        print("Post stored and images indexed!")


### Example 6: Check for Missed Images (Spider Recovery)

If the spider was down and an image wasn't ingested, you can:
1. Check if it exists in the database
2. Use reverse image search to find it if it exists under a different URL
3. Manually ingest it if it's truly missing


In [ ]:
# Example: Check for a specific image that might have been missed by the spider
# This is useful when you know an image should have been crawled but the spider was down

# Option 1: If you have the image URL, check directly
image_url = "https://i.redd.it/example-missed-image.jpg"  # Replace with actual URL

print("=== Checking if image exists in database ===")
result = reverse_search.check_if_crawled(image_url)

if result.found:
    print(f"Image FOUND in database!")
    print(f"Found in {len(result.matches)} location(s):")
    for match in result.matches:
        print(f"  - {match.match_type} match (confidence: {match.confidence:.2f})")
        if match.source_post_id:
            print(f"    From post: {match.source_post_id}")
else:
    print(f"Image NOT found in database.")
    print("\nThis could mean:")
    print("  1. The spider was down when this image was posted")
    print("  2. The image URL is different (try vector search)")
    print("  3. The image hasn't been crawled yet")
    
    # Try vector similarity search to find visually similar images
    if reverse_search.enable_vector_search:
        print("\n=== Trying vector similarity search ===")
        vector_result = reverse_search.find_exact_matches(image_url)
        if vector_result.matches:
            print(f"Found {len(vector_result.matches)} visually similar images:")
            for match in vector_result.matches:
                if match.match_type == "vector":
                    print(f"  - {match.image_url} (similarity: {match.confidence:.2f})")
                    if match.source_post_id:
                        print(f"    From post: {match.source_post_id}")


### Example 7: Manual Image Ingestion (Recovery)

If an image was missed by the spider, you can manually ingest it by:
1. Finding the source post/product URL
2. Using the crawler to fetch and store it
3. Indexing it for future search


In [ ]:
# Example: Manually ingest a missed image
# This is useful when you know the source URL but the spider missed it

# For Reddit posts:
from feed.platforms.reddit import RedditAdapter
from feed.storage.thread_storage import store_thread
from feed.utils.reddit_url_extractor import extract_post_id_from_url, parse_reddit_url

# Post URL that contains the missed image
post_url = "https://www.reddit.com/r/example/comments/abc123/missed_post/"

# Check if post already exists
post_id = extract_post_id_from_url(post_url)
check_query = """
MATCH (p:Post {id: $post_id})
RETURN p.id as id, p.title as title
LIMIT 1
"""
existing = neo4j.execute_read(check_query, parameters={"post_id": post_id})

if existing:
    print(f"Post {post_id} already exists in database")
    print(f"Title: {existing[0].get('title', 'N/A')}")
else:
    print(f"Post {post_id} not found. Fetching and storing...")
    
    # Fetch the post
    reddit = RedditAdapter()
    parsed = parse_reddit_url(post_url)
    
    if parsed and parsed.get('permalink'):
        permalink = parsed['permalink']
        post, comments, raw_post_data = reddit.fetch_thread(permalink)
        
        if post:
            # Extract images
            images = []
            post_images = reddit.extract_all_images(post, raw_post_data)
            for img_url in post_images:
                images.append({
                    "url": img_url,
                    "source": "post",
                    "post_id": post.id,
                })
            
            # Store in Neo4j
            store_thread(neo4j, post, comments, images, raw_post_data)
            print(f"Successfully stored post {post.id}")
            
            # Index images for future search
            print("Indexing images...")
            for img_url in post_images:
                reverse_search.index_image(img_url)
            print("Images indexed!")
        else:
            print("Failed to fetch post")
    else:
        print("Could not extract permalink from URL")


### Example 8: Batch Check for Missed Images

Check multiple image URLs at once to see which ones were missed:


In [ ]:
# Example: Batch check multiple images
image_urls = [
    "https://i.redd.it/image1.jpg",
    "https://i.redd.it/image2.jpg",
    "https://i.redd.it/image3.jpg",
    # Add your image URLs here
]

print(f"Checking {len(image_urls)} images...")
print("=" * 80)

found_images = []
missing_images = []

for img_url in image_urls:
    result = reverse_search.check_if_crawled(img_url)
    
    if result.found:
        found_images.append({
            "url": img_url,
            "matches": len(result.matches),
            "sources": [m.source_post_id for m in result.matches if m.source_post_id]
        })
        print(f"[FOUND] {img_url}")
        print(f"  Matches: {len(result.matches)}")
    else:
        missing_images.append(img_url)
        print(f"[MISSING] {img_url}")

print("\n" + "=" * 80)
print("Summary:")
print(f"  Found: {len(found_images)}")
print(f"  Missing: {len(missing_images)}")

if missing_images:
    print(f"\nMissing images (may need manual ingestion):")
    for url in missing_images:
        print(f"  - {url}")


### Example 9: Spider Recovery Tool

Use the SpiderRecovery service to check and recover missed images:


In [ ]:
from feed.services.spider_recovery import SpiderRecovery

# Initialize spider recovery service
recovery = SpiderRecovery(neo4j=neo4j, reverse_search=reverse_search)

# Check status of an image that might have been missed
image_url = "https://i.redd.it/example-missed-image.jpg"  # Replace with actual URL

print("=== Checking Image Status ===")
status = recovery.check_image_status(image_url, use_vector_search=True)

print(f"Image URL: {status['image_url']}")
print(f"Found in database: {status['found']}")

if status['found']:
    print(f"\nImage exists! Match type: {status['match_type']}")
    print(f"Found in {len(status['matches'])} location(s):")
    for match in status['matches']:
        print(f"  - {match['type']} match (confidence: {match['confidence']:.2f})")
        if match['source_post_id']:
            print(f"    From post: {match['source_post_id']}")
else:
    print("\nImage NOT found in database.")
    print("Recovery needed: Yes")
    
    if status['similar_images']:
        print(f"\nFound {len(status['similar_images'])} visually similar images:")
        for similar in status['similar_images']:
            print(f"  - {similar['url']} (similarity: {similar['similarity']:.2f})")
            if similar['source_post_id']:
                print(f"    From post: {similar['source_post_id']}")
    
    print("\nTo recover this image:")
    print("  1. Find the source post URL")
    print("  2. Use recovery.recover_post(post_url) to ingest it")


### Example 10: Recover Missed Post

If you know the post URL that contains a missed image, recover it:


In [ ]:
# Recover a missed post
post_url = "https://www.reddit.com/r/example/comments/abc123/missed_post/"

print("=== Recovering Missed Post ===")
result = recovery.recover_post(post_url, index_images=True)

print(f"Post URL: {result['post_url']}")
print(f"Success: {result['success']}")

if result['success']:
    if result.get('error') == "Post already exists in database":
        print("Post already exists - no recovery needed")
    else:
        print(f"Post ID: {result['post_id']}")
        print(f"Images found: {result['images_found']}")
        print(f"Images indexed: {result['images_indexed']}")
        print("Post successfully recovered and indexed!")
else:
    print(f"Recovery failed: {result.get('error', 'Unknown error')}")


## Video Identification & OSINT Research

This section demonstrates video identification using:
1. **Watermark Detection** - OCR to detect watermarks (e.g., "fantasyhd.com")
2. **Face Recognition** - Match faces against actor database
3. **Database Lookups** - Query data18.com and other sources
4. **Ontology Building** - Create graph structure for studios, actors, scenes
5. **Multi-Source Research** - Deep investigation across platforms


In [ ]:
from feed.services.video_watermark_detection import (
    VideoIdentificationService,
    VideoWatermarkDetector,
    Data18Crawler
)
from feed.storage.neo4j_connection import get_connection

# Connect to Neo4j
neo4j = get_connection()

# Initialize video identification service
video_service = VideoIdentificationService(
    neo4j=neo4j,
    enable_face_matching=True  # Enable face recognition
)

print("Video identification service initialized!")


### Example: Identify Video with Watermark

Identify a video file using watermark detection and OSINT research:


In [ ]:
# Example: Identify video with fantasyhd.com watermark
video_path = "~/Downloads/1766554370099356.webm"  # Replace with actual path

print("=== Video Identification ===")
print(f"Video: {video_path}")
print("\nStep 1: Detecting watermarks...")

# Initialize watermark detector
detector = VideoWatermarkDetector()
watermarks = detector.detect_watermarks_in_video(
    video_path,
    watermark_patterns=["fantasyhd.com", "fantasyhd"],
    sample_rate=30
)

print(f"Watermarks detected: {len(watermarks)}")
for wm in watermarks:
    print(f"  - '{wm.text}' (confidence: {wm.confidence:.2f}, frame: {wm.frame_number})")

# Extract studio from watermark
studio = None
if watermarks:
    watermark_text = watermarks[0].text.lower()
    if '.com' in watermark_text:
        studio = watermark_text.split('.com')[0]
        print(f"\nStep 2: Identified studio: {studio}")

# Full identification
if studio:
    print(f"\nStep 3: Running full identification...")
    result = video_service.identify_video(
        video_path,
        watermark_patterns=["fantasyhd.com"]
    )
    
    print(f"\n=== Identification Results ===")
    print(f"Identified: {result.identified}")
    print(f"Studio: {result.studio}")
    print(f"Watermarks: {len(result.watermarks)}")
    print(f"Face matches: {len(result.face_matches)}")
    print(f"Scene matches: {len(result.scene_matches)}")
    
    if result.scene_matches:
        print(f"\nScene Matches:")
        for scene in result.scene_matches[:5]:  # Show first 5
            print(f"  - {scene.scene_title}")
            print(f"    ID: {scene.scene_id}")
            print(f"    URL: {scene.url}")
            print(f"    Confidence: {scene.confidence:.2f}")
            print(f"    Match type: {scene.match_type}")


In [ ]:
# Example: Crawl fantasyhd actors from data18.com
crawler = Data18Crawler(delay_min=2.0, delay_max=5.0)

studio_slug = "fantasyhd"
print(f"=== Crawling {studio_slug} Actors from Data18 ===")

# Get all actors
actors = crawler.get_studio_actors(studio_slug)
print(f"Found {len(actors)} actors")

if actors:
    print(f"\nFirst 10 actors:")
    for actor in actors[:10]:
        print(f"  - {actor['name']} (ID: {actor['id']})")
        print(f"    URL: {actor['url']}")
    
    # Store in Neo4j
    print(f"\n=== Storing in Neo4j ===")
    
    # Create Studio node
    studio_query = """
    MERGE (s:Studio {slug: $slug})
    ON CREATE SET
        s.name = $name,
        s.slug = $slug,
        s.source_url = $source_url,
        s.created_at = datetime()
    RETURN s.slug as slug
    """
    neo4j.execute_write(
        studio_query,
        parameters={
            "slug": studio_slug,
            "name": studio_slug.title(),
            "source_url": f"https://www.data18.com/studios/{studio_slug}"
        }
    )
    print(f"Created/updated Studio: {studio_slug}")
    
    # Create Actor nodes
    stored = 0
    for actor in actors:
        actor_query = """
        MATCH (s:Studio {slug: $studio_slug})
        MERGE (a:Actor {data18_id: $actor_id})
        ON CREATE SET
            a.uuid = randomUUID(),
            a.name = $name,
            a.data18_id = $actor_id,
            a.profile_url = $url,
            a.created_at = datetime()
        ON MATCH SET
            a.updated_at = datetime()
        WITH a, s
        MERGE (s)-[:HAS_ACTOR]->(a)
        RETURN a.uuid as uuid
        """
        try:
            neo4j.execute_write(
                actor_query,
                parameters={
                    "studio_slug": studio_slug,
                    "actor_id": actor["id"],
                    "name": actor["name"],
                    "url": actor["url"]
                }
            )
            stored += 1
        except Exception as e:
            print(f"  Error storing actor {actor['name']}: {e}")
    
    print(f"Stored {stored}/{len(actors)} actors in Neo4j")
    print(f"\nGraph structure:")
    print(f"  Studio: {studio_slug}")
    print(f"  Actors: {stored} nodes")
    print(f"  Relationships: Studio -> HAS_ACTOR -> Actor")


In [ ]:
from feed.services.adult_content_crawlers import (
    MultiSourceCrawler,
    Data18Crawler,
    IAFDCrawler,
    IndexxxCrawler
)

# Initialize multi-source crawler
multi_crawler = MultiSourceCrawler(delay_min=2.0, delay_max=5.0)

studio = "fantasyhd"
print(f"=== Multi-Source Crawling for {studio} ===")

# Get performers from all sources
print("\n1. Fetching performers from Data18...")
data18_performers = multi_crawler.data18.get_studio_performers(studio)
print(f"   Found {len(data18_performers)} performers on Data18")

# Get scenes from IAFD
print("\n2. Fetching titles from IAFD...")
iafd_scenes = multi_crawler.iafd.get_distributor_titles("fantasyhd.com")
print(f"   Found {len(iafd_scenes)} titles on IAFD")

if iafd_scenes:
    print(f"\n   Sample IAFD titles:")
    for scene in iafd_scenes[:5]:
        print(f"     - {scene.title}")

# Search scenes across all sources
print("\n3. Searching scenes across all sources...")
all_scenes = multi_crawler.search_scenes_multi_source(studio=studio)
print(f"   Total unique scenes found: {len(all_scenes)}")

# Group by source
from collections import defaultdict
scenes_by_source = defaultdict(list)
for scene in all_scenes:
    scenes_by_source[scene.source].append(scene)

print(f"\n   Scenes by source:")
for source, scenes in scenes_by_source.items():
    print(f"     {source}: {len(scenes)} scenes")


### Example: Adult Reverse Image Search

Use adult-specific reverse image search tools for performer identification:


In [ ]:
from feed.services.video_watermark_detection import VideoWatermarkDetector
from feed.services.adult_content_crawlers import AdultReverseImageSearch
import cv2

# Extract frames from video for reverse search
video_path = "~/Downloads/1766554370099356.webm"  # Replace with actual path

print("=== Adult Reverse Image Search ===")
print(f"Video: {video_path}")

# Extract clear frames (focus on faces, avoid overlays)
detector = VideoWatermarkDetector()
frames = detector.extract_frames(video_path, sample_rate=60, max_frames=10)

print(f"\nExtracted {len(frames)} frames")

# Initialize adult reverse image search
reverse_search = AdultReverseImageSearch()

# Try reverse search on clear frames
print("\nPerforming reverse image search on clear frames...")
for i, (frame, frame_number, timestamp) in enumerate(frames[:3]):  # Limit to 3 frames
    print(f"\n  Frame {frame_number} (timestamp: {timestamp:.2f}s):")
    
    # Save frame temporarily
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
        cv2.imwrite(tmp.name, frame)
        
        # Search using adult reverse image search
        results = reverse_search.search_by_image(tmp.name, provider="namethatporn")
        
        if results:
            print(f"    Found {len(results)} potential matches:")
            for result in results[:3]:  # Show top 3
                print(f"      - Performer: {result.get('performer', 'N/A')}")
                print(f"        Scene: {result.get('scene', 'N/A')}")
                print(f"        Confidence: {result.get('confidence', 0):.2f}")
        else:
            print(f"    No matches found (placeholder - implement actual API)")


### Example: Stash Integration (Primary Source)

Stash is a self-hosted adult media organizer with:
- Perceptual hashing for scene identification
- StashDB integration (crowd-sourced metadata)
- Community scrapers for FantasyHD
- Built-in performer recognition
- Comprehensive scene matching

If you have Stash running locally, use it as the primary source:


In [ ]:
from feed.services.stash_integration import StashClient, StashIntegration

# Connect to your Stash instance
stash_url = "http://192.168.0.222:9999"  # Your Stash instance
# stash_api_key = "your_api_key"  # Optional, if API key is required

print(f"=== Stash Integration ===")
print(f"Connecting to: {stash_url}")

# Initialize Stash client
stash_client = StashClient(stash_url)

# Get FantasyHD scenes from Stash
print("\nFetching FantasyHD scenes from Stash...")
fantasyhd_scenes = stash_client.get_fantasyhd_scenes(limit=50)

print(f"Found {len(fantasyhd_scenes)} FantasyHD scenes in Stash")

if fantasyhd_scenes:
    print(f"\nSample scenes:")
    for scene in fantasyhd_scenes[:5]:
        print(f"  - {scene.title}")
        print(f"    ID: {scene.id}")
        print(f"    URL: {scene.url or f'{stash_url}/scenes/{scene.id}'}")
        print(f"    Date: {scene.date or 'N/A'}")
        print(f"    Performers: {', '.join([p.name for p in (scene.performers or [])])}")
        if scene.phash:
            print(f"    Phash: {scene.phash[:16]}...")
        print()


### Example: Video Identification with Stash

Use Stash as the primary source for video identification:


In [ ]:
# Example: Identify video using Stash integration
video_path = "~/Downloads/1766554370099356.webm"  # Replace with actual path
stash_url = "http://192.168.0.222:9999"  # Your Stash instance

print("=== Video Identification with Stash ===")

# Initialize video identification service with Stash
video_service = VideoIdentificationService(
    neo4j=neo4j,
    enable_face_matching=True,
    use_multi_source=True,
    stash_url=stash_url  # Enable Stash integration
)

# Identify video
result = video_service.identify_video(
    video_path,
    watermark_patterns=["fantasyhd.com"]
)

print(f"\nIdentification Results:")
print(f"  Identified: {result.identified}")
print(f"  Studio: {result.studio}")
print(f"  Watermarks: {len(result.watermarks)}")
print(f"  Scene matches: {len(result.scene_matches)}")

# Group scene matches by source
from collections import defaultdict
matches_by_source = defaultdict(list)
for scene in result.scene_matches:
    matches_by_source[scene.match_type].append(scene)

print(f"\nScene matches by source:")
for source, scenes in matches_by_source.items():
    print(f"  {source}: {len(scenes)} matches")
    for scene in scenes[:3]:  # Show first 3
        print(f"    - {scene.scene_title}")
        if scene.actors:
            print(f"      Actors: {', '.join(scene.actors[:3])}")
        print(f"      URL: {scene.url}")
        print(f"      Confidence: {scene.confidence:.2f}")

# Stash matches have highest confidence
stash_matches = [s for s in result.scene_matches if s.match_type == 'stash']
if stash_matches:
    print(f"\n=== Stash Matches (Highest Confidence) ===")
    for scene in stash_matches[:5]:
        print(f"  - {scene.scene_title}")
        print(f"    Stash URL: {scene.url}")
        print(f"    Confidence: {scene.confidence:.2f}")


### Example: Search Stash by Performer

Search for scenes by performer name in Stash:


In [ ]:
# Example: Search Stash for scenes by performer
stash_url = "http://192.168.0.222:9999"
performer_name = "Natalie"  # Example from your URL

print(f"=== Searching Stash for Performer: {performer_name} ===")

stash_client = StashClient(stash_url)

# Search scenes by performer
scenes = stash_client.search_scenes(
    performer_name=performer_name,
    studio="FantasyHD",
    limit=20
)

print(f"Found {len(scenes)} scenes with {performer_name} in FantasyHD")

if scenes:
    print(f"\nScenes:")
    for scene in scenes:
        print(f"  - {scene.title}")
        print(f"    Date: {scene.date or 'N/A'}")
        print(f"    URL: {scene.url or f'{stash_url}/scenes/{scene.id}'}")
        print(f"    Performers: {', '.join([p.name for p in (scene.performers or [])])}")
        if scene.stash_id:
            print(f"    StashDB ID: {scene.stash_id}")
        print()


## Forum Thread Parsing & Wanted Metadata Tracking

This section demonstrates:
1. **Forum Thread Parsing** - Parse vBulletin forums (planetsuzy.org)
2. **Image Extraction** - Extract images from threads for reverse search
3. **Wanted Metadata** - Track missing content from various sources
4. **Cross-Source Verification** - Verify performers across 18eighteen, data18, forums
5. **Evaluation Framework** - Benchmark testing for reverse lookups


In [ ]:
from feed.services.forum_parser import VBulletinParser
from feed.services.forum_storage import ForumStorage
from feed.services.wanted_metadata_tracker import WantedMetadataTracker
from feed.services.cross_source_verifier import CrossSourceVerifier

# Initialize services
forum_parser = VBulletinParser("http://www.planetsuzy.org")
forum_storage = ForumStorage(neo4j)
wanted_tracker = WantedMetadataTracker(neo4j)
cross_verifier = CrossSourceVerifier(neo4j)

print("Forum parsing and wanted tracking services initialized!")


### Example: Parse Forum Thread

Parse a vBulletin thread and extract images for reverse search:


In [ ]:
# Example: Parse planetsuzy.org thread about Natalie Monroe
thread_url = "http://www.planetsuzy.org/showthread.php?p=23377199&styleid=3"

print("=== Parsing Forum Thread ===")
print(f"URL: {thread_url}")

# Parse thread
thread = forum_parser.parse_thread(thread_url)

if thread:
    print(f"\nThread Information:")
    print(f"  Title: {thread.title}")
    print(f"  Thread ID: {thread.thread_id}")
    print(f"  Forum: {thread.forum_name}")
    print(f"  Author: {thread.author}")
    print(f"  Posts: {len(thread.posts or [])}")
    print(f"  Images: {len(thread.images or [])}")
    print(f"  Tags: {', '.join(thread.tags or [])}")
    
    # Show images
    if thread.images:
        print(f"\nImages found in thread:")
        for i, img_url in enumerate(thread.images[:10], 1):  # Show first 10
            print(f"  {i}. {img_url}")
    
    # Store in Neo4j
    print(f"\n=== Storing in Neo4j ===")
    success = forum_storage.store_thread(thread, index_images=True)
    
    if success:
        print("Thread stored successfully!")
        print("  - Forum node created")
        print("  - ForumThread node created")
        print("  - ForumPost nodes created")
        print("  - Image nodes created (for reverse search)")
        print("  - Tag nodes created")
        
        # Index images for reverse search
        print(f"\n=== Indexing Images for Reverse Search ===")
        for img_url in (thread.images or [])[:5]:  # Index first 5
            reverse_search.index_image(img_url)
        print("Images indexed!")
    else:
        print("Failed to store thread")
else:
    print("Failed to parse thread")


### Example: Track Wanted Metadata

Track missing content from various sources (18eighteen.com, etc.):


In [ ]:
# Example: Track wanted metadata for Natalie Monroe
performer_name = "Natalie Monroe"

# Source: 18eighteen.com (missing from self-hosted platform)
eighteen_url = "https://www.18eighteen.com/nude-teen-photos/Natalie-Monroe/43308/?nats=MTAwNC45LjMuMy43MDMuMC4wLjAuMA"

print("=== Tracking Wanted Metadata ===")
print(f"Performer: {performer_name}")
print(f"Source URL: {eighteen_url}")

# Create wanted entry
wanted_id = wanted_tracker.create_wanted_entry(
    performer_name=performer_name,
    source_url=eighteen_url,
    source_type="18eighteen",
    metadata={
        "photo_id": "43308",
        "category": "nude-teen-photos"
    }
)

print(f"Wanted entry created: {wanted_id}")

# Link forum thread to wanted entry
thread_url = "http://www.planetsuzy.org/showthread.php?p=23377199&styleid=3"
linked = wanted_tracker.link_forum_thread_to_wanted(thread_url, performer_name)
if linked:
    print(f"Linked forum thread to wanted entry")

# Cross-reference sources
print(f"\n=== Cross-Reference Sources ===")
cross_ref = wanted_tracker.cross_reference_sources(performer_name)

print(f"Wanted sources: {len(cross_ref.get('wanted_sources', []))}")
print(f"Scene IDs: {len(cross_ref.get('scene_ids', []))}")
print(f"Studios: {cross_ref.get('studios', [])}")
print(f"Has content: {cross_ref.get('has_content', False)}")
print(f"Is wanted: {cross_ref.get('is_wanted', False)}")


### Example: Cross-Source Verification

Verify if performer exists across multiple sources:


In [ ]:
# Example: Verify Natalie Monroe across sources
performer_name = "Natalie Monroe"
eighteen_url = "https://www.18eighteen.com/nude-teen-photos/Natalie-Monroe/43308/"

print("=== Cross-Source Verification ===")
print(f"Performer: {performer_name}")

# Check 18eighteen to data18 mapping
print(f"\n1. Checking 18eighteen → data18 mapping...")
mapping = cross_verifier.check_18eighteen_to_data18_mapping(
    performer_name,
    eighteen_url
)

print(f"   18eighteen URL: {mapping['18eighteen_url']}")
print(f"   Data18 URL: {mapping.get('data18_url', 'N/A')}")
print(f"   Data18 slug: {mapping.get('data18_slug', 'N/A')}")
print(f"   Mapping possible: {mapping.get('mapping_possible', False)}")
print(f"   Data18 found: {mapping.get('data18_found', False)}")

# Verify across all sources
print(f"\n2. Verifying across all sources...")
sources = ['data18', 'neo4j', 'stash']
verification = cross_verifier.verify_performer_across_sources(
    performer_name,
    sources
)

print(f"   Sources checked: {sources}")
for source, result in verification.get('sources', {}).items():
    print(f"   {source}:")
    if result.get('found'):
        print(f"     Found: Yes")
        if 'url' in result:
            print(f"     URL: {result['url']}")
        if 'scene_count' in result:
            print(f"     Scenes: {result['scene_count']}")
    else:
        print(f"     Found: No")


### Example: Evaluation Framework

Run benchmark tests for reverse lookup evaluation:


In [ ]:
from feed.services.evaluation_framework import (
    ReverseLookupEvaluator,
    TestCase,
    BenchmarkMetrics
)

# Initialize evaluator
evaluator = ReverseLookupEvaluator(neo4j, stash_url="http://192.168.0.222:9999")

# Create test cases
test_cases = [
    TestCase(
        test_id="test_001",
        test_type="image",
        input_data={
            "image_url": "https://i.redd.it/example1.jpg"
        },
        expected_result={
            "found": True,
            "min_confidence": 0.7
        }
    ),
    TestCase(
        test_id="test_002",
        test_type="video",
        input_data={
            "video_path": "~/Downloads/1766554370099356.webm",
            "watermark_patterns": ["fantasyhd.com"]
        },
        expected_result={
            "identified": True,
            "studio": "fantasyhd"
        }
    ),
    # Add more test cases...
]

print("=== Running Image Lookup Evaluation ===")
image_metrics = evaluator.evaluate_image_lookup(
    [tc for tc in test_cases if tc.test_type == "image"]
)

print(f"\nResults:")
print(f"  Total tests: {image_metrics.total_tests}")
print(f"  Passed: {image_metrics.passed_tests}")
print(f"  Failed: {image_metrics.failed_tests}")
print(f"  Accuracy: {image_metrics.accuracy:.2%}")
print(f"  Avg confidence: {image_metrics.average_confidence:.2f}")
print(f"  Avg time: {image_metrics.processing_time:.2f}s")
print(f"  Source breakdown: {image_metrics.source_breakdown}")

# Generate report
report = evaluator.generate_benchmark_report(image_metrics)
print(f"\n=== Benchmark Report ===")
print(report[:500] + "..." if len(report) > 500 else report)
